## RAG Fusion - 앙상블 검색 통합 기법

이전 두 노트북에서 확인한 한계에서 출발한다.

- [02_Agentic RAG](02_Agentic%20RAG%20-%20Agent%EA%B0%80%20%EA%B2%80%EC%83%89%20%EC%A0%84%EB%9E%B5%20%EC%9E%90%EC%9C%A8%20%EA%B2%B0%EC%A0%95.ipynb)는 질문마다 도구(벡터/그래프)를 **하나씩 순차적으로** 선택한다. 도구를 바꿔가며 LLM을 여러 번 호출해야 하므로 느리고, 처음 선택이 틀리면 재시도만큼 지연이 쌓인다.
- [03_Self-Corrective RAG](03_Self-Corrective%20RAG%20-%20%EA%B2%80%EC%83%89%20%EA%B2%B0%EA%B3%BC%20%EC%9E%90%EA%B8%B0%ED%8F%89%EA%B0%80%C2%B7%EC%9E%AC%EC%8B%9C%EB%8F%84.ipynb)는 한 번에 **하나의 질의·하나의 리트리버** 결과만 채점하고, 관련 문서가 하나도 없을 때만 재검색한다. 질문 자체가 모호해서 "그럭저럭 관련 있는" 문서 몇 개만 애매하게 잡히면 재검색 조건이 걸리지 않는다.

RAG Fusion은 다른 전략을 쓴다. "최선의 질의 하나"나 "최적의 도구 하나"를 고르려 하지 않고, **여러 개의 질의 변형과 여러 개의 검색 방식을 동시에 실행**한 뒤, 결과의 **순위(rank)**를 기준으로 점수를 합산하는 **Reciprocal Rank Fusion(RRF)**으로 하나의 순위표를 만든다. 개별 검색 하나가 놓친 문서를 다른 질의·다른 검색 방식이 채워주는 앙상블 구조다.

### 흐름
1. 원래 질문을 LLM으로 여러 개의 하위 질의(query variations)로 재작성한다.
2. 각 하위 질의로 벡터 검색을 실행해 질의 개수만큼의 순위 리스트를 얻는다.
3. RRF로 여러 순위 리스트를 하나로 융합한다.
4. (확장) 벡터 검색(dense)과 키워드 검색(BM25, sparse)처럼 **서로 다른 알고리즘**의 결과도 같은 방식으로 융합해, 표현이 다른 질문에도 견고하게 대응한다.
5. 융합된 상위 문서로 최종 답변을 생성한다.

```text
pip install langchain-classic langchain-community langchain-openai faiss-cpu
pip install rank_bm25   # BM25 키워드 검색 (하이브리드 앙상블 파트)
```


In [1]:
!pip install rank_bm25


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv

# .env 파일의 내용 불러오기
load_dotenv("C:/env/.env")


True

### [0] 공통 준비: LLM, 임베딩 모델

In [3]:
from typing import Dict, List, Tuple
from collections import defaultdict

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


c:\Users\storm\AppData\Local\Programs\Python\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_27148\1000150977.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


### [1] 샘플 도메인 재구성: 테크노바 지식베이스

01·02·03번 노트북과 동일한 "테크노바" 조직·프로젝트 사실 문장을 그대로 사용한다. 이 노트북만으로도
독립 실행이 되도록 벡터 스토어를 다시 만든다.

In [4]:
company_facts = [
    "김민준은 테크노바의 AI팀 소속이다.",
    "이서연은 AI팀의 팀장이다.",
    "AI팀은 '그래프 RAG 엔진' 프로젝트를 담당한다.",
    "박지훈은 데이터팀 소속이다.",
    "최유진은 데이터팀의 팀장이다.",
    "데이터팀은 '추천시스템 고도화' 프로젝트를 담당한다.",
    "AI팀은 데이터팀과 긴밀히 협업한다.",
    "정다은은 인프라팀 소속이다.",
    "한소희는 인프라팀의 팀장이다.",
    "인프라팀은 '검색 인프라 개선' 프로젝트를 담당한다.",
    "데이터팀은 인프라팀과 긴밀히 협업한다.",
    "'그래프 RAG 엔진' 프로젝트는 '검색 인프라 개선' 프로젝트의 결과물에 의존한다.",
    "박지훈은 '추천시스템 고도화' 프로젝트의 담당자이다.",
    "김민준은 '그래프 RAG 엔진' 프로젝트의 담당자이다.",
    "프로덕트팀은 '온보딩 자동화' 프로젝트를 담당한다.",
    "정다은은 '검색 인프라 개선' 프로젝트의 담당자이다.",
]

docs = [
    Document(page_content=fact, metadata={"fact_id": i})
    for i, fact in enumerate(company_facts)
]

vectorstore = FAISS.from_documents(docs, embeddings)


def format_docs(documents: List[Document]) -> str:
    return "\n".join(f"- {d.page_content}" for d in documents)


answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "아래 [사실] 목록만 근거로 질문에 답하라. 목록에 있는 여러 사실을 서로 연결해 "
            "논리적으로 답을 이끌어낼 수 있다면 그렇게 종합해서 답하라(예: '어떤 프로젝트의 "
            "담당자다'라는 사실은 그 프로젝트 주제와 관련된 일을 한다는 근거로 쓸 수 있다). "
            "[사실] 목록과 무관하거나, 목록을 조합해도 전혀 추론할 수 없는 경우에만 "
            "'주어진 사실만으로는 알 수 없다'라고 답하라.\n\n[사실]\n{context}",
        ),
        ("human", "{question}"),
    ]
)
answer_chain = answer_prompt | llm | StrOutputParser()

print(f"사실 문장 수: {len(company_facts)}")


사실 문장 수: 16


### [2] Naive RAG의 한계 재확인: 단일 질의의 사각지대

질문 하나를 그대로 임베딩해 top-k를 가져오는 방식은, 질문에 쓰인 표현과 정답 문서의 표현이 겹치지
않으면 후보 자체에서 밀려난다.

아래 질문은 "검색"이라는 단어 때문에 `'검색 인프라 개선'` 관련 문장은 쉽게 잡히지만, 같은 주제로
연결되는 `'그래프 RAG 엔진'` 담당자 정보는 "검색"이라는 단어를 직접 쓰지 않아 놓치기 쉽다. 질문을
어떻게 표현하느냐에 따라 top-k 안에 들어오는 문서가 달라진다.

In [5]:
def naive_rag(question: str, k: int = 3) -> dict:
    retrieved = vectorstore.similarity_search(question, k=k)
    answer = answer_chain.invoke(
        {"context": format_docs(retrieved), "question": question}
    )
    return {"retrieved": retrieved, "answer": answer}


probe_query = "테크노바에서 검색과 관련된 일을 하는 사람은 누구야?"

naive_result = naive_rag(probe_query)
print("=== Naive RAG 검색 결과 ===")
print(format_docs(naive_result["retrieved"]))
print("\n=== Naive RAG 답변 ===")
print(naive_result["answer"])


=== Naive RAG 검색 결과 ===
- 인프라팀은 '검색 인프라 개선' 프로젝트를 담당한다.
- 김민준은 테크노바의 AI팀 소속이다.
- '그래프 RAG 엔진' 프로젝트는 '검색 인프라 개선' 프로젝트의 결과물에 의존한다.

=== Naive RAG 답변 ===
주어진 사실만으로는 알 수 없다.


### [3] 질의 확장 (Multi-Query 생성) — RAG Fusion의 첫 축

원래 질문 하나만 검색에 쓰는 대신, LLM에게 **같은 의도를 다른 표현·다른 관점**으로 재작성한 질의를
여러 개 만들게 한다. 동의어, 상위 개념, 세부 키워드로 표현을 바꾸면 벡터 검색이 서로 다른 문서를
후보로 끌어온다.

`with_structured_output`으로 질의 리스트를 바로 받아, 파싱 없이 다음 단계로 넘긴다.

In [6]:
class QueryVariations(BaseModel):
    queries: List[str] = Field(
        description="원래 질문을 서로 다른 표현·관점으로 재작성한 검색 질의 목록"
    )


multi_query_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 검색 질의를 다양화하는 도우미다. 아래 질문을 벡터 검색에 쓸 수 있도록 "
            "서로 다른 표현·핵심어·관점으로 재작성한 질의를 {n}개 만들어라.\n"
            "- 원래 질문의 의도는 유지하되, 동의어·상위 개념·세부 키워드 등 표현을 다양화한다.\n"
            "- 각 질의는 한 문장으로 작성하고, 질의끼리 서로 겹치지 않게 한다.",
        ),
        ("human", "{question}"),
    ]
)
query_variation_generator = llm.with_structured_output(QueryVariations)
multi_query_chain = multi_query_prompt | query_variation_generator


def generate_query_variations(question: str, n: int = 4) -> List[str]:
    """원래 질문을 포함해 총 n+1개의 질의 리스트를 반환한다."""
    variations = multi_query_chain.invoke({"question": question, "n": n}).queries
    return [question] + variations


variations = generate_query_variations(probe_query, n=4)
print("=== 생성된 질의 변형 ===")
for i, q in enumerate(variations):
    label = "원본" if i == 0 else f"변형 {i}"
    print(f"[{label}] {q}")


=== 생성된 질의 변형 ===
[원본] 테크노바에서 검색과 관련된 일을 하는 사람은 누구야?
[변형 1] 테크노바에서 검색 업무를 담당하는 직원은 누구인가요?
[변형 2] 테크노바의 검색 관련 부서에서 일하는 사람은 어떤 직책을 가지고 있나요?
[변형 3] 테크노바에서 검색 서비스를 운영하는 팀의 구성원은 누구인지 알고 싶어요.
[변형 4] 테크노바에서 검색 관련 직무를 수행하는 인물은 누구인지 궁금합니다.


### [4] 질의별 벡터 검색 실행 → 여러 순위 리스트

각 질의로 독립적으로 벡터 검색을 실행한다. 질의 개수만큼의 순위 리스트(ranked list)가 생기고,
같은 문서가 여러 리스트에 서로 다른 순위로 등장할 수 있다.

In [7]:
def retrieve_for_queries(queries: List[str], k: int = 4) -> List[List[Document]]:
    """질의별로 벡터 검색을 실행해 순위 리스트 목록을 반환한다."""
    return [vectorstore.similarity_search(q, k=k) for q in queries]


ranked_lists = retrieve_for_queries(variations, k=4)

for q, docs_for_q in zip(variations, ranked_lists):
    print(f"질의: {q}")
    for rank, d in enumerate(docs_for_q, start=1):
        print(f"  {rank}위: {d.page_content}")
    print()


질의: 테크노바에서 검색과 관련된 일을 하는 사람은 누구야?
  1위: 인프라팀은 '검색 인프라 개선' 프로젝트를 담당한다.
  2위: 김민준은 테크노바의 AI팀 소속이다.
  3위: '그래프 RAG 엔진' 프로젝트는 '검색 인프라 개선' 프로젝트의 결과물에 의존한다.
  4위: 정다은은 '검색 인프라 개선' 프로젝트의 담당자이다.

질의: 테크노바에서 검색 업무를 담당하는 직원은 누구인가요?
  1위: 인프라팀은 '검색 인프라 개선' 프로젝트를 담당한다.
  2위: 정다은은 '검색 인프라 개선' 프로젝트의 담당자이다.
  3위: 김민준은 테크노바의 AI팀 소속이다.
  4위: '그래프 RAG 엔진' 프로젝트는 '검색 인프라 개선' 프로젝트의 결과물에 의존한다.

질의: 테크노바의 검색 관련 부서에서 일하는 사람은 어떤 직책을 가지고 있나요?
  1위: 인프라팀은 '검색 인프라 개선' 프로젝트를 담당한다.
  2위: 김민준은 테크노바의 AI팀 소속이다.
  3위: '그래프 RAG 엔진' 프로젝트는 '검색 인프라 개선' 프로젝트의 결과물에 의존한다.
  4위: 정다은은 '검색 인프라 개선' 프로젝트의 담당자이다.

질의: 테크노바에서 검색 서비스를 운영하는 팀의 구성원은 누구인지 알고 싶어요.
  1위: 인프라팀은 '검색 인프라 개선' 프로젝트를 담당한다.
  2위: 김민준은 테크노바의 AI팀 소속이다.
  3위: 정다은은 '검색 인프라 개선' 프로젝트의 담당자이다.
  4위: AI팀은 '그래프 RAG 엔진' 프로젝트를 담당한다.

질의: 테크노바에서 검색 관련 직무를 수행하는 인물은 누구인지 궁금합니다.
  1위: 인프라팀은 '검색 인프라 개선' 프로젝트를 담당한다.
  2위: 정다은은 '검색 인프라 개선' 프로젝트의 담당자이다.
  3위: 김민준은 테크노바의 AI팀 소속이다.
  4위: '그래프 RAG 엔진' 프로젝트는 '검색 인프라 개선' 프로젝트의 결과물에 의존한다.



### [5] Reciprocal Rank Fusion (RRF)

RRF는 **유사도 점수가 아니라 순위**만으로 여러 리스트를 합산한다.

$$\text{score}(d) = \sum_{l \in \text{lists}} \frac{1}{k + \text{rank}_l(d)}$$

- $\text{rank}_l(d)$는 리스트 $l$에서 문서 $d$의 순위(1부터 시작). 리스트에 없으면 더하지 않는다.
- $k$(기본 60)는 상위 순위의 영향력을 완화하는 damping 상수다. $k$가 작을수록 1위 문서의 가중치가
  커지고, 클수록 "몇 등이었나"보다 "여러 리스트에 걸쳐 등장했는가"의 비중이 커진다.
- 순위만 쓰기 때문에 코사인 유사도, BM25 점수처럼 **스케일이 전혀 다른 검색 방식의 결과도 그대로
  섞을 수 있다.** 이는 [7]의 하이브리드 검색에서 그대로 재사용된다.

In [8]:
def reciprocal_rank_fusion(
    ranked_lists: List[List[Document]],
    k: int = 60,
    top_n: int = 4,
) -> List[Tuple[Document, float]]:
    """여러 개의 순위 리스트를 RRF 점수로 합산해 하나의 순위로 융합한다.

    문서 동일성은 page_content로 판단한다(같은 텍스트를 다른 질의가 다시 찾아온 경우 중복 제거).
    """
    scores: Dict[str, float] = defaultdict(float)
    doc_by_key: Dict[str, Document] = {}

    for ranked_list in ranked_lists:
        for rank, doc in enumerate(ranked_list, start=1):
            key = doc.page_content
            scores[key] += 1.0 / (k + rank)
            doc_by_key.setdefault(key, doc)

    fused = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [(doc_by_key[key], score) for key, score in fused[:top_n]]


fused_result = reciprocal_rank_fusion(ranked_lists, top_n=4)

print("=== RRF 융합 결과 ===")
for rank, (doc, score) in enumerate(fused_result, start=1):
    print(f"{rank}위 (score={score:.4f}) {doc.page_content}")


=== RRF 융합 결과 ===
1위 (score=0.0820) 인프라팀은 '검색 인프라 개선' 프로젝트를 담당한다.
2위 (score=0.0801) 김민준은 테크노바의 AI팀 소속이다.
3위 (score=0.0794) 정다은은 '검색 인프라 개선' 프로젝트의 담당자이다.
4위 (score=0.0630) '그래프 RAG 엔진' 프로젝트는 '검색 인프라 개선' 프로젝트의 결과물에 의존한다.


### [6] RAG Fusion 파이프라인 완성 & Naive와 비교

[3]~[5]를 하나로 묶어 `rag_fusion` 함수로 만든다. 같은 질문에 대해 Naive RAG와 나란히 비교해,
질의 확장 + RRF가 단일 질의보다 더 넓고 정돈된 근거를 모으는지 확인한다.

In [9]:
def rag_fusion(
    question: str,
    n_variations: int = 4,
    k: int = 4,
    top_n: int = 4,
) -> dict:
    queries = generate_query_variations(question, n=n_variations)
    ranked_lists = retrieve_for_queries(queries, k=k)
    fused = reciprocal_rank_fusion(ranked_lists, top_n=top_n)
    fused_docs = [doc for doc, _ in fused]
    answer = answer_chain.invoke(
        {"context": format_docs(fused_docs), "question": question}
    )
    return {"queries": queries, "fused": fused, "answer": answer}


fusion_result = rag_fusion(probe_query)

print(f"질문: {probe_query}\n")
print("=== Naive RAG ===")
print(naive_result["answer"])
print("\n=== RAG Fusion (Multi-Query + RRF) ===")
print("융합된 근거:")
for doc, score in fusion_result["fused"]:
    print(f"  (score={score:.4f}) {doc.page_content}")
print(f"\n답변: {fusion_result['answer']}")


질문: 테크노바에서 검색과 관련된 일을 하는 사람은 누구야?

=== Naive RAG ===
주어진 사실만으로는 알 수 없다.

=== RAG Fusion (Multi-Query + RRF) ===
융합된 근거:
  (score=0.0820) 인프라팀은 '검색 인프라 개선' 프로젝트를 담당한다.
  (score=0.0801) 김민준은 테크노바의 AI팀 소속이다.
  (score=0.0796) 정다은은 '검색 인프라 개선' 프로젝트의 담당자이다.
  (score=0.0627) '그래프 RAG 엔진' 프로젝트는 '검색 인프라 개선' 프로젝트의 결과물에 의존한다.

답변: 정다은은 '검색 인프라 개선' 프로젝트의 담당자로, 이 프로젝트는 검색과 관련된 일을 하고 있다. 따라서 테크노바에서 검색과 관련된 일을 하는 사람은 정다은이다.


### [7] 앙상블 확장 — Dense(벡터) + Sparse(BM25) 검색 융합 (하이브리드 검색)

지금까지는 **같은 검색 알고리즘(벡터)**에 질의만 여러 개 던졌다. 이번에는 **서로 다른 검색
알고리즘**의 결과를 같은 RRF로 융합한다.

- **벡터 검색(dense)**: 의미적으로 유사한 문서를 찾는다. 표현이 달라도 의미가 통하면 잡아내지만,
  고유명사·특정 키워드가 그대로 안 들어가면 놓칠 수 있다.
- **BM25(sparse, 키워드 검색)**: 질의에 쓰인 단어가 문서에 그대로 등장하는지를 기준으로 점수를 매긴다.
  정확한 용어·고유명사 매칭에 강하지만, 동의어나 문장 구조가 다르면 놓친다.

두 방식은 서로 다른 이유로 실패하기 때문에, RRF로 결과를 합치면 한쪽이 놓친 문서를 다른 쪽이
채워주는 효과가 생긴다.

In [10]:
# pip install rank_bm25
from langchain_community.retrievers import BM25Retriever

BASE_K = 4

vector_retriever = vectorstore.as_retriever(search_kwargs={"k": BASE_K})
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = BASE_K


def hybrid_fuse(question: str, top_n: int = 4) -> List[Tuple[Document, float]]:
    """벡터(dense) 결과와 BM25(sparse) 결과를 RRF로 융합한다."""
    dense_docs = vector_retriever.invoke(question)
    sparse_docs = bm25_retriever.invoke(question)
    return reciprocal_rank_fusion([dense_docs, sparse_docs], top_n=top_n)


keyword_query = "'그래프 RAG 엔진' 프로젝트 담당자는 누구야?"

dense_only = vector_retriever.invoke(keyword_query)
sparse_only = bm25_retriever.invoke(keyword_query)
hybrid = hybrid_fuse(keyword_query)

print(f"질문: {keyword_query}\n")
print("=== 벡터 검색(dense) ===")
print(format_docs(dense_only))
print("\n=== BM25(sparse) ===")
print(format_docs(sparse_only))
print("\n=== 하이브리드 융합(RRF) ===")
for rank, (doc, score) in enumerate(hybrid, start=1):
    print(f"{rank}위 (score={score:.4f}) {doc.page_content}")


질문: '그래프 RAG 엔진' 프로젝트 담당자는 누구야?

=== 벡터 검색(dense) ===
- AI팀은 '그래프 RAG 엔진' 프로젝트를 담당한다.
- 김민준은 '그래프 RAG 엔진' 프로젝트의 담당자이다.
- '그래프 RAG 엔진' 프로젝트는 '검색 인프라 개선' 프로젝트의 결과물에 의존한다.
- 박지훈은 '추천시스템 고도화' 프로젝트의 담당자이다.

=== BM25(sparse) ===
- 김민준은 '그래프 RAG 엔진' 프로젝트의 담당자이다.
- AI팀은 '그래프 RAG 엔진' 프로젝트를 담당한다.
- '그래프 RAG 엔진' 프로젝트는 '검색 인프라 개선' 프로젝트의 결과물에 의존한다.
- 프로덕트팀은 '온보딩 자동화' 프로젝트를 담당한다.

=== 하이브리드 융합(RRF) ===
1위 (score=0.0325) AI팀은 '그래프 RAG 엔진' 프로젝트를 담당한다.
2위 (score=0.0325) 김민준은 '그래프 RAG 엔진' 프로젝트의 담당자이다.
3위 (score=0.0317) '그래프 RAG 엔진' 프로젝트는 '검색 인프라 개선' 프로젝트의 결과물에 의존한다.
4위 (score=0.0156) 박지훈은 '추천시스템 고도화' 프로젝트의 담당자이다.


### [8] LangChain 내장 `EnsembleRetriever`로 동일 패턴 재현

[7]에서 직접 만든 "여러 리트리버 결과를 RRF로 합치기"는 LangChain의 `EnsembleRetriever`가 그대로
제공한다. `weights`로 각 리트리버의 기여도를 조절할 수 있다는 점만 다르다.

| 수동 구현 | LangChain 래퍼 |
|---|---|
| `hybrid_fuse(question)` | `EnsembleRetriever(retrievers=[...])` |
| 리스트별 동일 가중치 | `weights=[...]`로 리트리버별 가중치 지정 가능 |
| `reciprocal_rank_fusion(...)` | 내부적으로 동일한 RRF 로직 사용 |

In [11]:
from langchain_classic.retrievers.ensemble import EnsembleRetriever

ensemble_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.5, 0.5],
)

ensemble_docs = ensemble_retriever.invoke(keyword_query)

print("=== LangChain EnsembleRetriever 결과 ===")
print(format_docs(ensemble_docs))


=== LangChain EnsembleRetriever 결과 ===
- AI팀은 '그래프 RAG 엔진' 프로젝트를 담당한다.
- 김민준은 '그래프 RAG 엔진' 프로젝트의 담당자이다.
- '그래프 RAG 엔진' 프로젝트는 '검색 인프라 개선' 프로젝트의 결과물에 의존한다.
- 박지훈은 '추천시스템 고도화' 프로젝트의 담당자이다.
- 프로덕트팀은 '온보딩 자동화' 프로젝트를 담당한다.


### [9] Naive vs RAG Fusion vs Hybrid Ensemble 종합 비교

여러 질문에 대해 세 방식의 답변을 나란히 비교한다.

In [12]:
def hybrid_rag(question: str) -> str:
    fused = hybrid_fuse(question)
    fused_docs = [doc for doc, _ in fused]
    return answer_chain.invoke(
        {"context": format_docs(fused_docs), "question": question}
    )


test_queries = [
    "테크노바에서 검색과 관련된 일을 하는 사람은 누구야?",
    "'그래프 RAG 엔진' 프로젝트 담당자는 누구야?",
    "인프라팀과 협업하는 팀이 담당하는 프로젝트는?",
]

for q in test_queries:
    naive_answer = naive_rag(q)["answer"]
    fusion_answer = rag_fusion(q)["answer"]
    hybrid_answer = hybrid_rag(q)

    print(f"질문: {q}")
    print(f"- Naive RAG (단일 질의)              : {naive_answer}")
    print(f"- RAG Fusion (Multi-Query + RRF)      : {fusion_answer}")
    print(f"- Hybrid Ensemble (벡터 + BM25, RRF)  : {hybrid_answer}")
    print("-" * 80)


질문: 테크노바에서 검색과 관련된 일을 하는 사람은 누구야?
- Naive RAG (단일 질의)              : 주어진 사실만으로는 알 수 없다.
- RAG Fusion (Multi-Query + RRF)      : 정다은은 '검색 인프라 개선' 프로젝트의 담당자로, 이 프로젝트는 검색과 관련된 일을 하고 있다. 따라서 테크노바에서 검색과 관련된 일을 하는 사람은 정다은이다.
- Hybrid Ensemble (벡터 + BM25, RRF)  : 주어진 사실만으로는 알 수 없다.
--------------------------------------------------------------------------------
질문: '그래프 RAG 엔진' 프로젝트 담당자는 누구야?
- Naive RAG (단일 질의)              : '그래프 RAG 엔진' 프로젝트의 담당자는 김민준이다.
- RAG Fusion (Multi-Query + RRF)      : '그래프 RAG 엔진' 프로젝트의 담당자는 김민준이다.
- Hybrid Ensemble (벡터 + BM25, RRF)  : '그래프 RAG 엔진' 프로젝트의 담당자는 김민준이다.
--------------------------------------------------------------------------------
질문: 인프라팀과 협업하는 팀이 담당하는 프로젝트는?
- Naive RAG (단일 질의)              : 인프라팀과 협업하는 팀은 데이터팀이며, 데이터팀은 인프라팀과 긴밀히 협업한다는 사실이 있습니다. 따라서 인프라팀과 협업하는 팀인 데이터팀은 '검색 인프라 개선' 프로젝트와 관련된 일을 한다고 추론할 수 있습니다.
- RAG Fusion (Multi-Query + RRF)      : 인프라팀은 '검색 인프라 개선' 프로젝트를 담당하고 있으며, 데이터팀이 인프라팀과 긴밀히 협업합니다. 따라서 인프라팀과 협업하는 데이터팀이 담당하는 프로젝트는 '검색 인프라 개선'

### [10] 정리

| 구분 | Naive RAG | Multi-Query RAG Fusion | Hybrid Ensemble (Dense+BM25) |
|---|---|---|---|
| 질의 수 | 1개(원 질문 그대로) | N+1개(원 질문 + 재작성 질의) | 1개(같은 질의를 서로 다른 알고리즘에 전달) |
| 검색 방식 | 벡터 검색 1회 | 벡터 검색 N+1회 | 벡터(dense) + BM25(sparse) |
| 결과 결합 | 없음(top-k 그대로) | RRF로 여러 순위 리스트 융합 | RRF로 두 알고리즘 결과 융합 |
| 강점 | 단순, 빠름 | 질문 표현이 애매해도 다양한 각도로 후보 확보 | 의미 유사성과 정확한 키워드 매칭을 모두 커버 |
| 약점 | 표현이 안 맞으면 후보 누락 | 질의 생성·검색 호출 수 증가(비용·지연) | BM25 품질이 토크나이저에 좌우(한국어는 형태소 분석기 권장) |

**구현 포인트**
- RRF는 유사도 "점수"가 아니라 "순위"만 사용하므로, 스케일이 전혀 다른 검색 방식(코사인 유사도 vs
  BM25 점수)의 결과도 그대로 융합할 수 있다.
- `k`(기본 60)는 순위 격차의 영향을 완화하는 damping 상수다. 작을수록 1위 문서의 가중치가 커지고,
  클수록 "여러 리스트에 걸쳐 나타났는가"가 중요해진다.
- 문서 동일성 판단(dedup 키)은 `page_content` 기준으로 했다. 실전에서는 문서 ID·청크 ID를 쓰는 편이
  더 안전하다.
- `EnsembleRetriever`는 동일한 RRF 로직을 `weights`로 각 리트리버의 기여도를 조절할 수 있게 감싼
  래퍼다.

**언제 쓰면 좋은가**
- 사용자 질문 표현이 다양하고 정형화되지 않은 서비스(챗봇 등)에서 벡터 검색의 재현율을 높이고 싶을 때
- 의미 검색과 키워드 검색이 서로 다른 실패 패턴을 보이는 도메인(고유명사·코드·에러 메시지 등 정확한
  문자열 매칭이 중요한 경우)
- 여러 검색 결과를 일단 하나의 후보 풀로 합친 뒤, [Reranking](../04_고급%20검색%20전략과%20품질%20최적화/03_Reranking%20(Cross-Encoder%EC%99%80%20Cohere%20Rerank).ipynb)으로 최종 순서를 다시 정리하고 싶을 때

**한계와 실전 확장 포인트**
- 질의 변형 개수(N)와 검색 k가 늘어날수록 임베딩·LLM 호출 비용이 커진다. 지연에 민감한 서비스는
  질의 개수를 3~5개로 제한하는 편이다.
- 이 노트북은 벡터+BM25만 융합했지만, [01_Graph RAG](01_Graph%20RAG%20-%20%EA%B4%80%EA%B3%84%ED%98%95%20%EA%B2%80%EC%83%89%20%EC%9B%90%EB%A6%AC.ipynb)의
  그래프 순회 결과나 [02_Agentic RAG](02_Agentic%20RAG%20-%20Agent%EA%B0%80%20%EA%B2%80%EC%83%89%20%EC%A0%84%EB%9E%B5%20%EC%9E%90%EC%9C%A8%20%EA%B2%B0%EC%A0%95.ipynb)의
  도구 결과도 순위 리스트 하나를 추가하는 형태로 같은 RRF에 포함시킬 수 있다.
- RRF는 "이미 검색된 것들의 순서"만 조정할 뿐, 애초에 어느 리스트에도 없는 정답은 만들어내지
  못한다. [03_Self-Corrective RAG](03_Self-Corrective%20RAG%20-%20%EA%B2%80%EC%83%89%20%EA%B2%B0%EA%B3%BC%20%EC%9E%90%EA%B8%B0%ED%8F%89%EA%B0%80%C2%B7%EC%9E%AC%EC%8B%9C%EB%8F%84.ipynb)의
  채점·재검색과 결합하면 융합 결과 자체의 관련성까지 검증할 수 있다.
- 기본 `BM25Retriever`는 공백 기준 토큰화라 한국어 조사·어미 변화에 약하다. 실전에서는
  `kiwipiepy` 등 형태소 분석기로 전처리한 뒤 BM25 인덱스를 구축하는 것이 좋다.
